# Notebook 4: Hypothesis Testing

**Difficulty:** ⭐⭐⭐⭐ (Upper Intermediate) | **Estimated Time:** 3 hours

---

## The Big Question

> *"District A houses seem cheaper than District B. But is that a real pattern, or just a coincidence of which houses we happened to survey?"*

This is the core problem of **hypothesis testing**: deciding whether an observed difference in data is a genuine signal or just random noise. It is the foundation of A/B testing, feature selection, model comparison, and scientific research.

---

### Topics Covered
1. The Courtroom Analogy: Null & Alternative Hypotheses
2. Type I & Type II Errors
3. Significance Level α and p-values
4. Two-sample t-test (comparing means of two groups)
5. Chi-square test (testing independence of categorical variables)
6. Full decision workflow

---

### Libraries Used
```
numpy, pandas, matplotlib, seaborn, scipy.stats
```

## 1. Why Does This Matter for ML?

Hypothesis testing shows up throughout the ML pipeline:

| ML Situation | Hypothesis Test |
|---|---|
| Is feature X predictive of house price? | t-test on correlation coefficient |
| Does model A outperform model B? | Paired t-test on validation scores |
| Is school district independent of house quality rating? | Chi-square test |
| Did the new recommendation algorithm improve click-through rate? | A/B test (two-sample t-test) |
| Are the coefficients in my regression model non-zero? | t-test on each coefficient |

When you read a paper that says "the improvement is statistically significant (p < 0.05)", they are reporting a hypothesis test. When scikit-learn's `SelectKBest` ranks features by importance, many of the underlying scores are hypothesis test statistics.

**Understanding hypothesis testing = understanding whether you can trust the patterns your model finds.**

## 2. The Courtroom Analogy

Hypothesis testing works exactly like a criminal trial:

| Trial | Hypothesis Test |
|---|---|
| Defendant is **innocent until proven guilty** | **Null hypothesis H₀** is assumed true until data proves otherwise |
| Prosecution must present evidence | You must collect data that argues against H₀ |
| "Beyond reasonable doubt" standard | Significance level α (e.g., 5% chance of being wrong) |
| Verdict: Guilty or Not Guilty | Decision: Reject H₀ or Fail to Reject H₀ |

### Key Language

**H₀ (Null Hypothesis):** The boring, safe, "nothing interesting is happening" claim.
- Example: *"House prices in District A and District B are the same."*
- The default assumption — you assume this until data says otherwise

**H₁ (Alternative Hypothesis):** The interesting claim you are trying to prove.
- Example: *"House prices in District A and District B are different."*
- You are building the case for this

**Critical Point:** You never "prove" H₀ is true. A verdict of "Not Guilty" does not mean the defendant is innocent — it means the evidence was insufficient to convict. Similarly, "failing to reject H₀" does not mean H₀ is true — it means your data wasn't strong enough to disprove it.

## 3. Two Kinds of Mistakes

In any decision under uncertainty, two errors are possible:

|  | H₀ is Actually True | H₀ is Actually False |
|---|---|---|
| **We Reject H₀** | ❌ Type I Error (False Positive) | ✅ Correct! |
| **We Fail to Reject H₀** | ✅ Correct! | ❌ Type II Error (False Negative) |

### Type I Error (False Positive)
- **Court analogy:** Convicting an innocent person
- **ML analogy:** Concluding a useless feature is important; deploying a model that isn't actually better
- **Probability:** α (the significance level you choose)
- Choosing α = 0.05 means: "I accept a 5% chance of making this mistake"

### Type II Error (False Negative)
- **Court analogy:** Acquitting a guilty person
- **ML analogy:** Failing to detect a genuinely better model; missing a truly important feature
- **Probability:** β (related to statistical power = 1 − β)
- Increased sample size reduces β

### The Trade-off
Reducing α (being stricter) reduces Type I errors but increases Type II errors. There is no free lunch — you must choose your acceptable error rates based on the cost of each mistake in your application.

In [ ]:
# ── Setup: import all libraries ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Reproducibility
np.random.seed(42)

# Clean, consistent plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

## 4. The Two-Sample t-test

### When to Use
Use a two-sample t-test when you want to compare the **means** of two independent groups.

**Our question:** Are house prices in District A significantly different from District B?

### Assumptions
1. The two samples are **independent** (houses in District A are different buildings from District B)
2. The data within each group is roughly **normally distributed** — OR the sample size is large enough that the CLT applies (n ≥ 30 per group)
3. The groups have **similar variances** (for the standard pooled t-test; there are variants that relax this)

### Plain English
> We compute how large the observed difference in means is, *relative to the noise* in the data. If the signal-to-noise ratio is large, we conclude the difference is real.

### The t-statistic Formula

$$t = \frac{\bar{x}_1 - \bar{x}_2}{SE_{\text{pooled}}}$$

Where the pooled standard error is:
$$SE_{\text{pooled}} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}$$

- **Numerator** = observed difference in means (the "signal")
- **Denominator** = how much noise/variability we'd expect just from sampling
- **Large |t|** = signal is large relative to noise → evidence against H₀
- **Small |t|** = difference could easily be explained by random sampling

In [ ]:
# ── Create two districts with different mean house prices ─────────────────────

N_PER_DISTRICT = 100   # number of houses surveyed in each district

# District A: lower-priced area (mean $280K)
# District B: higher-priced area (mean $320K)
# Both have the same std ($60K) — same variability, just different centers
MEAN_A, MEAN_B = 280_000, 320_000
STD_BOTH       = 60_000

prices_A = np.random.normal(loc=MEAN_A, scale=STD_BOTH, size=N_PER_DISTRICT)
prices_B = np.random.normal(loc=MEAN_B, scale=STD_BOTH, size=N_PER_DISTRICT)

# Round to nearest dollar for realism
prices_A = np.round(prices_A, 0)
prices_B = np.round(prices_B, 0)

# Ensure no negative prices (houses can't cost less than $0)
prices_A = np.clip(prices_A, 50_000, None)
prices_B = np.clip(prices_B, 50_000, None)

print("=== District Summaries ===")
print(f"  District A: mean=${np.mean(prices_A):,.0f}  std=${np.std(prices_A, ddof=1):,.0f}  n={len(prices_A)}")
print(f"  District B: mean=${np.mean(prices_B):,.0f}  std=${np.std(prices_B, ddof=1):,.0f}  n={len(prices_B)}")
print(f"\n  Observed difference in means: ${np.mean(prices_B) - np.mean(prices_A):,.0f}")
print(f"  True difference (population): ${MEAN_B - MEAN_A:,.0f}")
print()
print("Question: Is the observed $40K difference real, or just sampling noise?")

In [ ]:
# ── Visualise the two distributions and their overlap ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('House Prices: District A vs District B', fontsize=14, fontweight='bold')

# ── Left: Overlapping histograms ──
ax = axes[0]
ax.hist(prices_A / 1000, bins=25, alpha=0.6, color='steelblue',
        edgecolor='white', density=True, label=f'District A (mean=${np.mean(prices_A)/1000:.0f}K)')
ax.hist(prices_B / 1000, bins=25, alpha=0.6, color='coral',
        edgecolor='white', density=True, label=f'District B (mean=${np.mean(prices_B)/1000:.0f}K)')

# Mark each group's mean with a vertical line
ax.axvline(np.mean(prices_A) / 1000, color='steelblue', linestyle='--', linewidth=2)
ax.axvline(np.mean(prices_B) / 1000, color='coral',    linestyle='--', linewidth=2)

# Shade the overlap region to visualise how much they mix
x_range = np.linspace(50, 600, 400)
pdf_A = stats.norm.pdf(x_range * 1000, np.mean(prices_A), np.std(prices_A, ddof=1))
pdf_B = stats.norm.pdf(x_range * 1000, np.mean(prices_B), np.std(prices_B, ddof=1))
overlap = np.minimum(pdf_A, pdf_B) / 1000   # scale to density units
ax.fill_between(x_range, overlap, alpha=0.3, color='purple', label='Overlap region')

ax.set_xlabel('Price ($K)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Overlapping Distributions: Is the Gap Real?', fontsize=11)
ax.legend(fontsize=9)

# ── Right: Box plots for a cleaner comparison ──
ax2 = axes[1]
data_for_boxplot = pd.DataFrame({
    'Price ($K)': np.concatenate([prices_A / 1000, prices_B / 1000]),
    'District': ['A'] * len(prices_A) + ['B'] * len(prices_B)
})
sns.boxplot(data=data_for_boxplot, x='District', y='Price ($K)',
            palette=['steelblue', 'coral'], ax=ax2, width=0.5)
sns.stripplot(data=data_for_boxplot, x='District', y='Price ($K)',
              palette=['steelblue', 'coral'], ax=ax2, alpha=0.25, size=3)
ax2.set_title('Box Plot Comparison', fontsize=11)
ax2.set_xlabel('District', fontsize=11)

# Annotate the difference
diff = (np.mean(prices_B) - np.mean(prices_A)) / 1000
ax2.annotate('', xy=(1, np.mean(prices_B) / 1000), xytext=(0, np.mean(prices_A) / 1000),
             xycoords=('data', 'data'), textcoords=('data', 'data'),
             arrowprops=dict(arrowstyle='<->', color='black', lw=2))
ax2.text(0.5, (np.mean(prices_A) + np.mean(prices_B)) / 2 / 1000,
         f'  Δ=${diff:.0f}K', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Step-by-Step: Computing the t-statistic Manually

Before using `scipy.stats`, let's compute every step by hand. This builds the intuition for what the test actually does.

### Hypotheses
- **H₀:** μ_A = μ_B (prices in the two districts are the same)
- **H₁:** μ_A ≠ μ_B (prices are different; this is a two-tailed test)
- **α = 0.05** (we accept a 5% chance of a false positive)

### Decision Rule
We will **reject H₀** if |t| > t_critical, where t_critical comes from the t-distribution with the appropriate degrees of freedom.

In [ ]:
# ── Manual t-test calculation (step by step) ──────────────────────────────────

# Step 1: Sample statistics for each district
n1    = len(prices_A)
n2    = len(prices_B)
mean1 = np.mean(prices_A)
mean2 = np.mean(prices_B)
var1  = np.var(prices_A, ddof=1)   # sample variance (ddof=1 = Bessel's correction)
var2  = np.var(prices_B, ddof=1)

print("Step 1 — Sample Statistics")
print(f"  District A: n={n1}, mean=${mean1:,.0f}, var=${var1:,.0f}")
print(f"  District B: n={n2}, mean=${mean2:,.0f}, var=${var2:,.0f}")

# Step 2: Compute the Pooled Standard Error
# SE_pooled = sqrt( var1/n1 + var2/n2 )
# This measures: "how much noise do we expect in the difference of means?"
se_pooled = np.sqrt(var1 / n1 + var2 / n2)
print(f"\nStep 2 — Pooled Standard Error")
print(f"  SE_pooled = sqrt(var_A/n_A + var_B/n_B) = ${se_pooled:,.0f}")

# Step 3: Compute the t-statistic
# t = (mean1 - mean2) / SE_pooled
# Signal (difference in means) ÷ Noise (SE_pooled)
t_statistic = (mean1 - mean2) / se_pooled
print(f"\nStep 3 — t-statistic")
print(f"  t = ({mean1:,.0f} - {mean2:,.0f}) / {se_pooled:,.0f}")
print(f"  t = {mean1 - mean2:,.0f} / {se_pooled:,.0f}")
print(f"  t = {t_statistic:.4f}")

# Step 4: Degrees of freedom (Welch-Satterthwaite approximation)
# This accounts for unequal variances; for equal n with similar variance,
# df ≈ n1 + n2 - 2
df_numerator   = (var1 / n1 + var2 / n2) ** 2
df_denominator = (var1 / n1) ** 2 / (n1 - 1) + (var2 / n2) ** 2 / (n2 - 1)
df_welch       = df_numerator / df_denominator
print(f"\nStep 4 — Degrees of Freedom (Welch)")
print(f"  df = {df_welch:.2f}  (approx {n1 + n2 - 2} for equal-variance case)")

# Step 5: p-value — probability of seeing |t| this large or larger if H0 were true
# For a two-tailed test, multiply by 2 (we care about differences in both directions)
# stats.t.sf(x, df) = P(T > x) for the t-distribution ("survival function")
p_value_manual = 2 * stats.t.sf(abs(t_statistic), df=df_welch)
print(f"\nStep 5 — p-value")
print(f"  p = 2 × P(T > |{t_statistic:.4f}|) = {p_value_manual:.6f}")

# Step 6: Critical value for α=0.05 (two-tailed)
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha / 2, df=df_welch)   # 97.5th percentile
print(f"\nStep 6 — Critical Value")
print(f"  For α={alpha} (two-tailed), t_critical = ±{t_critical:.4f}")

In [ ]:
# ── Verify with scipy and make the decision ───────────────────────────────────

# scipy.stats.ttest_ind computes the Welch t-test (equal_var=False is the default)
# equal_var=False uses Welch's approximation (does NOT assume equal variances)
# equal_var=True would use the pooled t-test (assumes equal variances)
t_scipy, p_scipy = stats.ttest_ind(prices_A, prices_B, equal_var=False)

print("=== scipy.stats Verification ===")
print(f"  scipy t-statistic : {t_scipy:.4f}")
print(f"  Manual t-statistic: {t_statistic:.4f}")
print(f"  Match: {'YES' if abs(t_scipy - t_statistic) < 0.001 else 'NO'}")
print()
print(f"  scipy p-value : {p_scipy:.6f}")
print(f"  Manual p-value: {p_value_manual:.6f}")
print()

# Decision
print("=== DECISION ===")
print(f"  |t| = {abs(t_statistic):.4f}  vs  t_critical = {t_critical:.4f}")
print(f"  p-value = {p_scipy:.6f}  vs  α = {alpha}")
print()

if p_scipy < alpha:
    print("  Result: REJECT H₀")
    print(f"  Plain English: The data provides strong evidence that District A and")
    print(f"  District B have different mean house prices (p={p_scipy:.4f} < α={alpha}).")
    print(f"  The observed ${abs(mean1 - mean2) / 1000:.0f}K difference is unlikely to be")
    print(f"  due to chance alone.")
else:
    print("  Result: FAIL TO REJECT H₀")
    print(f"  Plain English: We do not have sufficient evidence to conclude the")
    print(f"  districts have different prices (p={p_scipy:.4f} ≥ α={alpha}).")
    print(f"  The observed difference could plausibly be due to random sampling.")

In [ ]:
# ── Visualise the t-distribution with critical regions ───────────────────────
# This is the core visual for understanding hypothesis testing:
# - The curve shows what t-values would look like if H0 were true
# - Red shaded areas = critical regions (only 5% of the curve is in these tails)
# - If our observed t falls in the red zone → reject H0

fig, ax = plt.subplots(figsize=(12, 6))

# t-distribution curve for our degrees of freedom
df_plot  = int(df_welch)
x        = np.linspace(-5, 5, 500)
y        = stats.t.pdf(x, df=df_plot)   # probability density function

ax.plot(x, y, 'k-', linewidth=2.5, label=f't-distribution (df={df_plot})', zorder=3)

# Shade the critical regions (α/2 = 2.5% in each tail)
x_left  = x[x <= -t_critical]
x_right = x[x >= t_critical]
ax.fill_between(x_left,  stats.t.pdf(x_left,  df_plot), alpha=0.4,
                color='crimson', label=f'Rejection region (each tail = {alpha/2:.1%})')
ax.fill_between(x_right, stats.t.pdf(x_right, df_plot), alpha=0.4,
                color='crimson')

# Shade the "do not reject" region in the middle
x_mid = x[(x >= -t_critical) & (x <= t_critical)]
ax.fill_between(x_mid, stats.t.pdf(x_mid, df_plot), alpha=0.15,
                color='steelblue', label=f'Do NOT reject H₀ region ({1-alpha:.0%})')

# Mark the critical values
ax.axvline(-t_critical, color='crimson', linestyle='--', linewidth=1.5,
           label=f't_critical = ±{t_critical:.3f}')
ax.axvline( t_critical, color='crimson', linestyle='--', linewidth=1.5)

# Mark our observed t-statistic
ax.axvline(t_statistic, color='darkgreen', linestyle='-', linewidth=3,
           label=f'Our t = {t_statistic:.3f}')

# Annotate the t-statistic position
ax.text(t_statistic - 0.15, max(y) * 0.6,
        f't = {t_statistic:.3f}\np = {p_scipy:.4f}',
        fontsize=10, color='darkgreen', fontweight='bold',
        ha='right', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Annotate the critical regions
ax.text(-4.2, 0.02, f'Reject H₀\n(α/2={alpha/2:.2f})', ha='center',
        fontsize=9, color='crimson', fontweight='bold')
ax.text( 4.2, 0.02, f'Reject H₀\n(α/2={alpha/2:.2f})', ha='center',
        fontsize=9, color='crimson', fontweight='bold')
ax.text(0, max(y) * 0.5, f'Do NOT reject H₀\n({1-alpha:.0%} of the distribution)',
        ha='center', fontsize=10, color='steelblue', fontweight='bold')

ax.set_title(
    f't-Distribution Under H₀: "District A prices = District B prices"\n'
    f'Observed t={t_statistic:.3f} | p={p_scipy:.4f} | α={alpha}',
    fontsize=12
)
ax.set_xlabel('t-statistic value', fontsize=11)
ax.set_ylabel('Probability Density', fontsize=11)
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(-5, 5)

plt.tight_layout()
plt.show()

decision_text = 'REJECT H₀' if p_scipy < alpha else 'FAIL TO REJECT H₀'
print(f"Decision: {decision_text}")
print(f"Our t={t_statistic:.3f} falls {'INSIDE' if abs(t_statistic) > t_critical else 'OUTSIDE'}"
      " the rejection region (red shaded area).")

## 6. Understanding the p-value

The **p-value** is one of the most misunderstood concepts in statistics.

### What p-value IS:
> The probability of observing a test statistic as extreme (or more extreme) than what we got, **assuming H₀ is true**.

If p = 0.0001: "If districts truly had the same prices, there's only a 0.01% chance we'd see this large a difference just from sampling."

### What p-value IS NOT:
- ❌ "The probability that H₀ is true" (you can't compute that without Bayesian priors)
- ❌ "The probability that our result is due to chance" (this rephrasing is subtly wrong)
- ❌ "A measure of the size or importance of the effect" (a tiny difference can be "significant" with enough data)

### The α Threshold
- α = 0.05 is a convention, not a law of nature
- In medical research: α = 0.001 (false positives can harm patients)
- In exploratory ML: α = 0.10 might be acceptable (you'll validate findings anyway)
- The key is deciding α **before** looking at the data

In [ ]:
# ── Demonstrate p-value intuitively via permutation ───────────────────────────
# If H0 is true (no real difference), we can simulate what t-values look like
# by randomly shuffling the district labels and recomputing t each time.
# Our observed t should be "extreme" relative to these null-distribution t values.

# Combine both districts into one pool
all_prices = np.concatenate([prices_A, prices_B])
n_total    = len(all_prices)

N_PERMUTATIONS = 5000   # number of simulated "null" experiments
null_t_values  = np.zeros(N_PERMUTATIONS)

for i in range(N_PERMUTATIONS):
    # Randomly shuffle: pretend district labels were assigned arbitrarily
    shuffled     = np.random.permutation(all_prices)
    fake_group_A = shuffled[:n1]    # first n1 go to "A"
    fake_group_B = shuffled[n1:]    # remainder go to "B"

    # Compute t for this fake assignment
    t_null, _ = stats.ttest_ind(fake_group_A, fake_group_B, equal_var=False)
    null_t_values[i] = t_null

# The permutation p-value: fraction of null t-values more extreme than observed
perm_p_value = np.mean(np.abs(null_t_values) >= abs(t_statistic))

print(f"Permutation test p-value : {perm_p_value:.4f}")
print(f"Parametric t-test p-value: {p_scipy:.4f}")
print("(They should be similar — both measure the same thing.)")

# ── Visualise the permutation distribution ──
fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(null_t_values, bins=60, density=True, color='steelblue',
        edgecolor='white', alpha=0.75, label='Null distribution\n(if H₀ were true)')

# Mark our observed t-value
ax.axvline(t_statistic, color='red', linewidth=3,
           label=f'Observed t = {t_statistic:.3f}')
ax.axvline(-abs(t_statistic), color='red', linewidth=3, linestyle='--')

# Shade the region more extreme than our observed t
extreme_left  = null_t_values[null_t_values <= t_statistic]
extreme_right = null_t_values[null_t_values >= abs(t_statistic)]
ax.axvspan(min(null_t_values), t_statistic, alpha=0.2, color='red',
           label=f'More extreme region\n(permutation p ≈ {perm_p_value:.4f})')
ax.axvspan(abs(t_statistic), max(null_t_values), alpha=0.2, color='red')

ax.set_title(
    'Permutation Test: What t-values Would Look Like If H₀ Were True',
    fontsize=12
)
ax.set_xlabel('t-statistic', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print()
print("Interpretation: Under H₀ (shuffled labels), t-values are centred at 0.")
print(f"Our observed t={t_statistic:.3f} is far out in the tail — very unlikely under H₀.")

## 7. The Chi-Square Test

### When to Use
Use a chi-square test when you want to test whether two **categorical** variables are independent.

**Our question:** Is house quality rating (Low / Medium / High) independent of school district?

This is a different kind of question from the t-test — we are not comparing means, we are comparing **frequencies** (counts of how many houses fall into each category combination).

### Plain English
> If school district and quality rating are **independent**, then knowing a house is in District A tells you nothing about whether it is Low/Medium/High quality. The quality ratings should appear in roughly the same proportions in both districts.
> 
> If they are **dependent**, then District A might have more High-quality homes while District B has more Low-quality homes — a systematic pattern.

### The Chi-Square Formula

$$\chi^2 = \sum_{\text{all cells}} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

Where:
- **O_ij** = observed count in cell (row i, column j)
- **E_ij** = expected count if the variables were independent = (row total × column total) / grand total

**Large χ²** = observed counts are far from expected → evidence that the variables are NOT independent

In [ ]:
# ── Create categorical data: house quality × school district ──────────────────
# We'll create two scenarios:
#   Scenario 1: quality IS dependent on district (there is a real association)
#   Scenario 2: quality is INDEPENDENT of district (no real association)

N_HOUSES = 300   # 300 houses in each district for this analysis

# --- Scenario 1: Dependent (association exists) ---
# District A: lower-priced area → more Low-quality, fewer High-quality
# District B: higher-priced area → more High-quality, fewer Low-quality
quality_A_dep = np.random.choice(['Low', 'Medium', 'High'],
                                  size=N_HOUSES,
                                  p=[0.50, 0.35, 0.15])   # mostly Low quality
quality_B_dep = np.random.choice(['Low', 'Medium', 'High'],
                                  size=N_HOUSES,
                                  p=[0.15, 0.35, 0.50])   # mostly High quality

# Build a contingency table using pandas crosstab
# A contingency table shows the frequency of each combination
district_col_dep = ['A'] * N_HOUSES + ['B'] * N_HOUSES
quality_col_dep  = list(quality_A_dep) + list(quality_B_dep)

df_dep = pd.DataFrame({'District': district_col_dep, 'Quality': quality_col_dep})

# Order the quality levels sensibly (Low → Medium → High)
df_dep['Quality'] = pd.Categorical(df_dep['Quality'],
                                    categories=['Low', 'Medium', 'High'],
                                    ordered=True)

contingency_dep = pd.crosstab(df_dep['District'], df_dep['Quality'],
                               margins=True, margins_name='Total')

print("=== Contingency Table (Scenario 1: Dependent) ===")
print(contingency_dep)
print()
print("Visual check: District A has many more Low-quality homes,")
print("District B has many more High-quality homes → looks dependent.")

In [ ]:
# ── Manual chi-square computation (step by step) ─────────────────────────────

# Work with the observed counts (without the margin row/column)
obs_table = pd.crosstab(df_dep['District'], df_dep['Quality'])
obs_array = obs_table.values   # convert to numpy array for arithmetic

print("Observed counts:")
print(obs_table)
print()

# Step 1: Compute expected frequencies
# If variables are independent:
#   E[i,j] = (row_i_total × col_j_total) / grand_total
# This is what we'd expect if district had NO effect on quality
grand_total = obs_array.sum()
row_totals  = obs_array.sum(axis=1, keepdims=True)   # sums per district
col_totals  = obs_array.sum(axis=0, keepdims=True)   # sums per quality level

expected = (row_totals * col_totals) / grand_total

expected_df = pd.DataFrame(expected,
                            index=obs_table.index,
                            columns=obs_table.columns)
print("Expected counts (if independent):")
print(expected_df.round(1))
print()
print("Compare observed vs expected row by row:")
diff_df = obs_table - expected_df
print("Observed - Expected:")
print(diff_df.round(1))

# Step 2: Compute chi-square statistic
# For each cell: (O - E)^2 / E
# Then sum all cells
cell_contributions = ((obs_array - expected) ** 2) / expected
chi2_manual        = cell_contributions.sum()

print(f"\nStep 2 — Chi-square statistic")
print(f"  χ² = Σ [(O-E)² / E] = {chi2_manual:.4f}")

# Step 3: Degrees of freedom
# df = (number of rows - 1) × (number of columns - 1)
df_chi2 = (obs_array.shape[0] - 1) * (obs_array.shape[1] - 1)
print(f"\nStep 3 — Degrees of freedom")
print(f"  df = ({obs_array.shape[0]}-1) × ({obs_array.shape[1]}-1) = {df_chi2}")

# Step 4: p-value
p_chi2_manual = 1 - stats.chi2.cdf(chi2_manual, df=df_chi2)
print(f"\nStep 4 — p-value")
print(f"  p = P(χ² > {chi2_manual:.4f}) = {p_chi2_manual:.8f}")

In [ ]:
# ── Verify with scipy and visualise ──────────────────────────────────────────

# scipy.stats.chi2_contingency takes the raw contingency table (no margins)
chi2_scipy, p_chi2_scipy, dof, expected_scipy = stats.chi2_contingency(obs_array)

print("=== scipy Verification ===")
print(f"  scipy χ²  : {chi2_scipy:.4f}")
print(f"  Manual χ² : {chi2_manual:.4f}")
print(f"  scipy p   : {p_chi2_scipy:.8f}")
print(f"  Manual p  : {p_chi2_manual:.8f}")
print()

alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, df=df_chi2)
print(f"  χ² critical (df={df_chi2}, α={alpha}): {chi2_critical:.4f}")

if p_chi2_scipy < alpha:
    print(f"\n  REJECT H₀: House quality and school district are NOT independent.")
    print(f"  Plain English: There is strong statistical evidence that the quality")
    print(f"  distribution of houses differs between District A and District B")
    print(f"  (p={p_chi2_scipy:.2e} << α={alpha}).")

# ── Visual: heatmap of observed vs expected ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Chi-Square Test: Observed vs Expected Counts', fontsize=13, fontweight='bold')

cmap = 'YlOrRd'
sns.heatmap(obs_array, annot=True, fmt='d', cmap=cmap, ax=axes[0],
            xticklabels=['Low', 'Medium', 'High'], yticklabels=['Dist A', 'Dist B'],
            cbar=False)
axes[0].set_title('Observed Counts', fontsize=11)

sns.heatmap(expected.round(0).astype(int), annot=True, fmt='d', cmap='YlGn', ax=axes[1],
            xticklabels=['Low', 'Medium', 'High'], yticklabels=['Dist A', 'Dist B'],
            cbar=False)
axes[1].set_title('Expected Counts (if independent)', fontsize=11)

residuals = (obs_array - expected) / np.sqrt(expected)   # standardised residuals
sns.heatmap(residuals, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[2],
            xticklabels=['Low', 'Medium', 'High'], yticklabels=['Dist A', 'Dist B'],
            cbar=True)
axes[2].set_title('Standardised Residuals\n(large |value| → big discrepancy)', fontsize=11)

plt.tight_layout()
plt.show()

print()
print("Standardised residuals > 2 or < -2 indicate cells driving the chi-square result.")

## 8. The Full Hypothesis Testing Workflow

Here is the complete process in one place. Follow this every time.

```
Step 1: STATE HYPOTHESES
   H₀: [null claim — what you assume by default]
   H₁: [alternative — what you are trying to prove]

Step 2: SET SIGNIFICANCE LEVEL
   Choose α BEFORE looking at data (e.g., α = 0.05)

Step 3: CHOOSE THE RIGHT TEST
   Comparing two means (continuous)?  → two-sample t-test
   Comparing categorical frequencies? → chi-square test
   Paired before/after measurements?  → paired t-test
   More than 2 groups?                → ANOVA

Step 4: COLLECT DATA & CHECK ASSUMPTIONS
   Sample size large enough? Independent? Data appropriate for the test?

Step 5: COMPUTE TEST STATISTIC
   t-test:  t = (mean1 - mean2) / SE_pooled
   chi²: χ² = Σ (O-E)²/E

Step 6: FIND THE p-VALUE
   p = P(seeing a statistic this extreme | H₀ is true)

Step 7: MAKE A DECISION
   If p < α: Reject H₀ (evidence supports H₁)
   If p ≥ α: Fail to Reject H₀ (insufficient evidence)

Step 8: STATE CONCLUSION IN PLAIN ENGLISH
   Translate back to the real-world question. Never just say "p < 0.05".
```

In [ ]:
# ── Complete workflow demonstration: end-to-end analysis ─────────────────────
# Scenario: A real estate developer claims that adding a school in a district
# raises average house prices above $300K.
# We have a sample of 60 houses from that district. Does the data support the claim?

# --- Step 1: State Hypotheses ---
print("STEP 1: State Hypotheses")
print("  H₀: μ = $300,000  (mean price is $300K, no improvement from school)")
print("  H₁: μ > $300,000  (mean price is ABOVE $300K — one-tailed test)")
print("  (One-tailed because the developer specifically claims prices are HIGHER)")
print()

# --- Step 2: Set significance level ---
alpha_one = 0.05
print(f"STEP 2: Set α = {alpha_one}")
print()

# --- Step 3: Choose the test ---
print("STEP 3: Choose test → one-sample t-test (comparing sample mean to known value)")
print()

# --- Step 4: Collect data ---
# Simulate: the district actually has mean ~$315K (school DID help a bit)
sample_new_district = np.random.normal(loc=315_000, scale=55_000, size=60)
print(f"STEP 4: Collect data")
print(f"  n = {len(sample_new_district)} houses sampled")
print(f"  Sample mean = ${np.mean(sample_new_district):,.0f}")
print(f"  Sample std  = ${np.std(sample_new_district, ddof=1):,.0f}")
print()

# --- Step 5 & 6: Compute test statistic and p-value ---
# scipy.stats.ttest_1samp: one-sample t-test against a known population mean
t_one, p_two_tailed = stats.ttest_1samp(sample_new_district, popmean=300_000)
# Convert two-tailed p to one-tailed p (we are testing μ > 300K, not ≠ 300K)
# One-tailed p = half of two-tailed p IF the direction matches our hypothesis
p_one_tailed = p_two_tailed / 2 if t_one > 0 else 1.0

print(f"STEP 5-6: Compute t-statistic and p-value")
print(f"  t = {t_one:.4f}")
print(f"  p (one-tailed) = {p_one_tailed:.4f}")
print()

# --- Step 7: Decision ---
print("STEP 7: Decision")
if p_one_tailed < alpha_one:
    decision = "REJECT H₀"
    print(f"  p={p_one_tailed:.4f} < α={alpha_one} → {decision}")
else:
    decision = "FAIL TO REJECT H₀"
    print(f"  p={p_one_tailed:.4f} ≥ α={alpha_one} → {decision}")
print()

# --- Step 8: Plain English conclusion ---
print("STEP 8: Plain English Conclusion")
if decision == "REJECT H₀":
    print(f"  The sample provides statistically significant evidence (p={p_one_tailed:.4f})")
    print(f"  that mean house prices in this district exceed $300K.")
    print(f"  This is consistent with the developer's claim about the school's impact.")
    print(f"  NOTE: This does NOT prove the school CAUSED the price increase — only that")
    print(f"  prices ARE higher than $300K on average.")
else:
    print(f"  We do not have sufficient evidence to conclude prices exceed $300K.")
    print(f"  The developer's claim is not supported by this sample.")

In [ ]:
# ── Bonus: Multiple Testing Problem ──────────────────────────────────────────
# If you run MANY hypothesis tests, some will be false positives by chance.
# This is the "multiple comparisons problem" — critical to understand for ML.

print("=== The Multiple Testing Problem ===")
print()
print("Suppose you test 20 house features for correlation with price.")
print("You use α=0.05 for each test.")
print("Even if NONE of the features are truly correlated,")
print("how many spurious 'significant' results do you expect?")
print()

n_tests     = 20
alpha_each  = 0.05
expected_fp = n_tests * alpha_each

print(f"Expected false positives = {n_tests} tests × {alpha_each} = {expected_fp:.0f}")
print()
print("One test per feature sounds fine. But with 20 tests, you expect ~1 false positive!")
print()

# Simulate this: 20 tests, all H0 true, α=0.05
np.random.seed(99)
n_simulations  = 10_000
false_positive_counts = []

for _ in range(n_simulations):
    # Simulate 20 t-tests where H0 is true (both groups from same distribution)
    false_positives = 0
    for __ in range(n_tests):
        g1 = np.random.normal(0, 1, 50)   # same distribution
        g2 = np.random.normal(0, 1, 50)   # same distribution → H0 is true
        _, p = stats.ttest_ind(g1, g2)
        if p < alpha_each:
            false_positives += 1
    false_positive_counts.append(false_positives)

false_positive_counts = np.array(false_positive_counts)

print(f"Simulation results over {n_simulations:,} repetitions:")
print(f"  Average false positives per experiment : {false_positive_counts.mean():.2f}  (expected {expected_fp:.1f})")
print(f"  Experiments with AT LEAST 1 false pos  : {np.mean(false_positive_counts >= 1)*100:.1f}%")
print(f"  Experiments with 2+ false positives    : {np.mean(false_positive_counts >= 2)*100:.1f}%")
print()
print("Solution: Bonferroni correction → use α* = α / n_tests = 0.05/20 = 0.0025")
print("This keeps the FAMILY-WISE error rate at 5% across all 20 tests combined.")

# Histogram of false positives
fig, ax = plt.subplots(figsize=(9, 4))
unique, counts = np.unique(false_positive_counts, return_counts=True)
ax.bar(unique, counts / n_simulations * 100, color='steelblue',
       edgecolor='black', width=0.6)
ax.set_xlabel('Number of False Positives', fontsize=11)
ax.set_ylabel('% of Experiments', fontsize=11)
ax.set_title(f'Distribution of False Positives\n'
             f'(20 tests × α=0.05, all H₀ true, n={n_simulations:,} simulations)',
             fontsize=11)
for u, c in zip(unique, counts):
    ax.text(u, c / n_simulations * 100 + 0.5, f'{c/n_simulations*100:.1f}%',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 9. t-test vs Chi-Square: Choosing the Right Test

| Feature | Two-Sample t-test | Chi-Square Test |
|---|---|---|
| **What it tests** | Are two group **means** different? | Are two categorical variables **independent**? |
| **Data type** | Continuous (e.g., price in dollars) | Categorical (e.g., Low/Med/High) |
| **Test statistic** | t = difference / SE | χ² = Σ (O-E)²/E |
| **Null hypothesis** | μ₁ = μ₂ | Variables are independent |
| **House price example** | "Is mean price in Dist A = mean price in Dist B?" | "Is quality rating independent of school district?" |
| **Assumption** | Normal or large n (CLT) | Expected counts ≥ 5 in each cell |

### Quick Decision Guide

```
What kind of data are you comparing?
│
├── Continuous outcome (price, area, income)
│   ├── 2 groups? → Two-sample t-test
│   ├── 2 paired measurements (before/after)? → Paired t-test
│   └── 3+ groups? → ANOVA
│
└── Categorical outcome (quality, type, colour)
    ├── Testing independence of 2 categorical variables? → Chi-square
    └── Comparing proportion to known value? → Binomial test / z-test for proportion
```

## 10. Summary: Key Takeaways

| Concept | Plain English | Technical |
|---|---|---|
| **H₀** | "Nothing interesting is happening" | Default assumption; requires evidence to reject |
| **H₁** | "Something real is happening" | What we're trying to show |
| **α** | "I accept being wrong X% of the time" | Significance level; set BEFORE seeing data |
| **p-value** | "If H₀ were true, how surprising is our data?" | P(data this extreme \| H₀ true) |
| **Type I Error** | Convicting the innocent | False positive; rate = α |
| **Type II Error** | Acquitting the guilty | False negative; rate = β |
| **t-test** | "Are the means of two groups different?" | Compares continuous group means |
| **Chi-square** | "Are these two categories related?" | Tests independence of categorical variables |
| **Multiple testing** | 20 tests × 5% chance = ~1 false positive | Bonferroni: use α/n per test |

### Connection to ML
- Every linear regression coefficient comes with a t-test built in
- Feature selection methods are often based on statistical tests
- A/B testing is just a two-sample t-test with a web experiment
- Model comparison ("is Model A better than Model B?") uses paired t-tests on validation scores

Next up: **Notebook 5** — Correlation, Covariance, and Linear Regression

## 11. Self-Check Questions

Test your understanding before moving on.

---

**Q1.** You set α = 0.05 and get p = 0.03. Should you reject H₀? What does this mean in plain English?

<details>
<summary>Click to reveal answer</summary>

**Yes, reject H₀.** Since p = 0.03 < α = 0.05, the result is statistically significant.

In plain English: "If the null hypothesis were true (no real difference between districts), there would only be a 3% chance of observing a difference as large as (or larger than) what we measured. Since we set our threshold at 5%, this is surprising enough to conclude there IS a real difference — we reject the idea that prices are the same."

Important nuance: This does NOT mean there is a 97% probability that H₁ is true, or that the difference is large or practically important. Statistical significance just means the evidence is strong enough to exceed your chosen threshold.
</details>

---

**Q2.** Your friend says: "We proved houses in District A are cheaper." Is this correct? What is the more accurate statement?

<details>
<summary>Click to reveal answer</summary>

**Not quite.** Three problems with this statement:

1. **We did not "prove" anything.** Hypothesis testing can never prove a claim — it can only provide evidence for or against it. We either reject or fail to reject H₀.

2. **We rejected H₀ (equality), not proved H₁.** The correct statement is: "The data provides statistically significant evidence (p < 0.05) that District A and District B have different mean house prices, with District A appearing lower."

3. **Correlation ≠ causation.** Even if prices really are lower, we haven't established WHY. It could be confounding variables (older buildings, distance from city centre, etc.).

More accurate: "We found a statistically significant difference in mean house prices between District A (\$280K) and District B (\$320K) using a two-sample t-test (t=–X.XX, p=0.00X). This is consistent with District A having lower prices, though further analysis is needed to understand the causes."
</details>

---

**Q3.** What is the difference between a t-test and a chi-square test? Give a house-price example for each.

<details>
<summary>Click to reveal answer</summary>

**Two-sample t-test:** Used when comparing the **means** of a **continuous** variable across two groups.
- Example: "Is the mean sale price of 3-bedroom houses (\$320K) significantly different from 4-bedroom houses (\$390K)?"
- Tests: H₀: μ(3-bed) = μ(4-bed)

**Chi-square test:** Used when testing whether two **categorical** variables are **independent** of each other.
- Example: "Is house condition rating (Poor/Fair/Good/Excellent) independent of whether the house sold above asking price (Yes/No)?"
- Tests: H₀: condition and sale-above-asking are independent

The key distinction: t-test = continuous outcome; chi-square = categorical outcome. If you try to run a t-test on categories or a chi-square on continuous data, you'll get meaningless results.
</details>

---

**Q4.** If you run 20 hypothesis tests with α = 0.05, expecting all H₀ to be true, how many false positives do you expect? What can you do about it?

<details>
<summary>Click to reveal answer</summary>

**Expected false positives = 20 × 0.05 = 1 false positive.**

Moreover, the probability of getting AT LEAST ONE false positive across 20 tests is:
P(at least 1 FP) = 1 − (1 − 0.05)²⁰ = 1 − 0.95²⁰ ≈ **64%**

This is the **multiple comparisons problem** (also called data dredging or p-hacking).

**Solutions:**
1. **Bonferroni correction:** Use α* = 0.05/20 = 0.0025 for each individual test. This keeps the family-wise error rate at 5%.
2. **Benjamini-Hochberg procedure:** Less conservative; controls the False Discovery Rate (FDR) instead of family-wise error rate.
3. **Pre-registration:** Specify your hypotheses before looking at data to prevent post-hoc cherry-picking.
4. **Validation:** Always validate "significant" findings on a held-out dataset.

In ML: this is why you shouldn't select features by p-value alone — always validate on held-out data.
</details>